# 🔬 OBJ-2 Structure Agent
## Environment: Python 3.10.20 · meeko_env

**Handles intents:** `TARGET_STRUCTURE`, `TARGET_POCKET`, `TARGET_DRUGGABILITY`,
`TARGET_LIGAND`, `TARGET_DOCKING`, `TARGET_VIRTUAL_SCREENING`, `TARGET_TO_LIGAND_DISCOVERY`

**Also called by:** Orchestrator after Obj-1 for `DISEASE_DRUGGABLE_TARGETS` / `FULL_PIPELINE`

**Reads:** `shared/kg_output.json` (written by Obj-1) — optional  
**Writes:** `shared/structure_output.json`

### ⚠️ Run this in the `meeko_env` (Python 3.10.20) kernel only
Meeko, AutoDock Vina, P2Rank, RDKit are only available here.

In [ ]:
# Run once — install LangChain in meeko_env
%pip install langchain langchain-core langgraph
# DO NOT install ollama here — we call it via HTTP REST API

In [1]:
P2RANK_PATH = r"C:\Dissertation\p2rank_2.5.1\prank.bat"   # ← your actual P2Rank path
VINA        = r"C:\Dissertation\vina.exe"                   # ← your actual Vina path

## ─── CELL A: All original Obj-2 imports (UNCHANGED) ───

In [2]:
# =========================
# CELL 1 — SETUP (UNCHANGED from your original Obj-2)
# =========================
import os, json, re, requests, subprocess, time, operator
import pandas as pd
from collections import Counter
from typing import TypedDict, Annotated, Optional, Any

# BioPython
from Bio.PDB import PDBParser, PDBIO, Select

# RDKit
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Descriptors
RDLogger.DisableLog('rdApp.*')

# Meeko — ONLY available in meeko_env / Python 3.10
from meeko import MoleculePreparation, PDBQTWriterLegacy

# LangChain — NEW
from langchain.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, END

# =========================
# PATHS — edit here only
# =========================
BASE_DIR      = os.getcwd()
STRUCTURE_DIR = os.path.join(BASE_DIR, "structures")
P2RANK_DIR    = os.path.join(BASE_DIR, "p2rank_output")
DOCKING_DIR   = os.path.join(BASE_DIR, "final_results")
for d in [STRUCTURE_DIR, P2RANK_DIR, DOCKING_DIR]: os.makedirs(d, exist_ok=True)

P2RANK_PATH   = r"C:\Dissertation\p2rank_2.5.1\prank.bat"  # ← your path
VINA          = r"C:\Dissertation\vina.exe"                  # ← your path

# Shared I/O
SHARED_DIR           = os.path.join(BASE_DIR, "shared")
os.makedirs(SHARED_DIR, exist_ok=True)
KG_INPUT_FILE        = os.path.join(SHARED_DIR, "kg_output.json")
STRUCTURE_OUTPUT_FILE = os.path.join(SHARED_DIR, "structure_output.json")

# Ollama HTTP endpoint — running in Obj-1's env, accessible via localhost
OLLAMA_URL  = "http://localhost:11434/api/chat"
LLM_MODEL   = "llama3:8b"

print("✅ Obj-2 setup complete  |  Python 3.10.20 / meeko_env")

✅ Obj-2 setup complete  |  Python 3.10.20 / meeko_env


## ─── CELL B: LLM via Ollama HTTP (no ollama package required) ───

In [3]:
def call_ollama_http(system: str, user: str, model: str = LLM_MODEL) -> str:
    """
    Call Ollama via HTTP REST API.
    Works without the `ollama` Python package — safe for meeko_env.
    Ollama server must be running: `ollama serve`
    """
    payload = {
        "model": model,
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
        "stream": False, "options": {"temperature": 0}
    }
    try:
        resp = requests.post(OLLAMA_URL, json=payload, timeout=120)
        resp.raise_for_status()
        return resp.json()["message"]["content"]
    except Exception as e:
        print(f"⚠️ Ollama HTTP call failed: {e}")
        return "[LLM unavailable — ensure Ollama server is running: ollama serve]"


print("✅ Ollama HTTP caller ready  (make sure 'ollama serve' is running)")

✅ Ollama HTTP caller ready  (make sure 'ollama serve' is running)


In [4]:
# ══════════════════════════════════════════════════════════════
# NATURAL LANGUAGE QUERY NORMALIZER (meeko_env version)
# Uses Ollama HTTP API — no ollama package needed
# ══════════════════════════════════════════════════════════════

def normalize_natural_query(query: str) -> str:
    """
    Converts informal / conversational queries into clean structured ones.
    Runs BEFORE entity extraction so natural phrasing works correctly.
    """
    system = """You are a biomedical query normalizer.
Convert informal or conversational user input into a clean, direct biomedical query."""

    user = f"""
Convert the following user input into a clean, direct biomedical query.

Rules:
- Keep the same meaning and intent
- Remove filler ("I want to", "can you", "tell me", "help me", "please", "i am curious", etc.)
- Preserve all biomedical entity names (diseases, proteins, drugs, gene names) EXACTLY
- Preserve UniProt IDs (e.g. P07471) unchanged
- Preserve SMILES strings unchanged
- Do NOT add new entities or change disease names
- If already clean, return unchanged

Examples:
"i want to know binding pockets of EGFR" → "Find binding pockets for EGFR"
"can you show me pockets of P07471" → "Find binding pockets for P07471"
"help me find drug candidates for egfr" → "Find drug candidates for EGFR"
"i need ligands that bind to p07471" → "Find ligands for P07471"
"dock something to egfr" → "Perform docking for EGFR"
"is tp53 druggable" → "Is TP53 druggable?"
"show pockets and find molecules that bind EGFR" → "Find binding pockets and ligands for EGFR"
"do virtual screening on P07471" → "Perform virtual screening for P07471"
"what small molecules could bind to egfr" → "Find ligands for EGFR"
"screen compounds against P07471" → "Perform virtual screening for P07471"

User input:
{query}

Return ONLY the normalized query. No explanation.
"""
    normalized = call_ollama_http(system, user).strip()
    if len(normalized) > len(query) * 3:
        return query
    print(f"   🔄 Normalized: {query!r} → {normalized!r}")
    return normalized


print("✅ Natural language normalizer loaded (meeko_env / HTTP API)")


✅ Natural language normalizer loaded (meeko_env / HTTP API)


## ─── CELL C: All original Obj-2 functions (COMPLETELY UNCHANGED) ───

In [5]:
# ══════════════════════════════════════════════════════════════
# CELL 2 — SAFE API REQUESTS (UNCHANGED)
# ══════════════════════════════════════════════════════════════

def safe_request(url, retries=3, delay=2, return_json=True):
    for i in range(retries):
        try:
            response = requests.get(url, timeout=15)
            if response.status_code == 200:
                return response.json() if return_json else response
            print(f"⚠️ Status Code {response.status_code} for: {url}")
        except requests.exceptions.RequestException as e:
            print(f"⚠️ Request failed (attempt {i+1}/{retries}): {e}")
        time.sleep(delay)
    print(f"❌ Failed after {retries} retries: {url}")
    return None


def safe_post_request(url, json_data, retries=3, delay=2):
    for i in range(retries):
        try:
            response = requests.post(url, json=json_data, timeout=15)
            if response.status_code == 200: return response.json()
            print(f"⚠️ POST Status {response.status_code}")
        except requests.exceptions.RequestException as e:
            print(f"⚠️ POST failed (attempt {i+1}): {e}")
        time.sleep(delay)
    print(f"❌ POST failed: {url}")
    return None


print("✅ API helpers loaded")

✅ API helpers loaded


In [6]:
# ══════════════════════════════════════════════════════════════
# STRUCTURE DOWNLOAD (UNCHANGED)
# ══════════════════════════════════════════════════════════════

def fetch_best_pdb(uniprot_id):
    """
    Select the best PDB structure for a UniProt ID from RCSB.
    Scoring criteria (higher = better):
      +30  X-ray crystallography (most reliable for docking)
      +20  Has real drug-like ligand (not just water/ions)
      +15  Resolution < 2.0 Angstrom
      +10  Resolution < 2.5 Angstrom
      +5   Resolution < 3.0 Angstrom
      -10  NMR / other methods (no resolution)
    Falls back to AlphaFold if no PDB found or all scores <= 0.
    """
    url   = "https://search.rcsb.org/rcsbsearch/v2/query"
    query = {
        "query": {"type": "terminal", "service": "text",
                  "parameters": {
                      "attribute": "rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers.database_accession",
                      "operator": "exact_match", "value": uniprot_id}},
        "return_type": "entry", "request_options": {"return_all_hits": True}
    }
    res = safe_post_request(url, query)
    if res is None: return None
    pdb_ids = [r["identifier"] for r in res.get("result_set", [])]
    if not pdb_ids:
        print(f"   No PDB entries found for {uniprot_id}")
        return None

    print(f"   Found {len(pdb_ids)} PDB entries — selecting best...")

    # Atoms/ions to ignore when checking for real ligands
    BAD_LIGANDS = {"HOH","NA","CL","K","MG","CA","ZN","SO4","PO4","GOL",
                   "EDO","PEG","DMS","MPD","FMT","ACT","IOD","BR","FE","CU","MN"}

    best_pdb, best_score, best_info = None, -999, {}

    for pdb_id in pdb_ids[:30]:   # check top 30 entries
        data = safe_request(f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}")
        if data is None: continue

        score = 0
        info  = {"pdb_id": pdb_id}

        # ── Experiment type ──
        try:
            exp_method = data["rcsb_entry_info"]["experimental_method"]
            info["method"] = exp_method
            if "X-RAY" in exp_method.upper():
                score += 30    # X-ray is best for docking
            elif "NMR" in exp_method.upper():
                score -= 10    # NMR has no resolution — penalise
            elif "ELECTRON" in exp_method.upper():
                score += 10    # cryo-EM acceptable
        except:
            pass

        # ── Resolution ──
        try:
            resolution = data["rcsb_entry_info"]["resolution_combined"][0]
            info["resolution"] = resolution
            if resolution < 2.0:   score += 15
            elif resolution < 2.5: score += 10
            elif resolution < 3.0: score += 5
            elif resolution > 3.5: score -= 10   # poor resolution
        except:
            info["resolution"] = None

        # ── Real ligands (drug-like molecules) ──
        try:
            lig_ids = data["rcsb_entry_container_identifiers"]["non_polymer_entity_ids"]
            real_ligs = [l for l in lig_ids if l not in BAD_LIGANDS]
            info["real_ligands"] = real_ligs
            if real_ligs: score += 20   # has drug-like ligand → great for docking
        except:
            info["real_ligands"] = []

        info["score"] = score
        print(f"   {pdb_id}: method={info.get('method','?')}, "
              f"res={info.get('resolution','?')}, "
              f"ligands={info.get('real_ligands',[])}, score={score}")

        if score > best_score:
            best_score = score
            best_pdb   = pdb_id
            best_info  = info

    if best_pdb:
        print(f"\n   Best PDB selected: {best_pdb} (score={best_score}, "
              f"resolution={best_info.get('resolution','?')}, "
              f"ligands={best_info.get('real_ligands',[])})")
    else:
        print("   No suitable PDB found")

    return best_pdb



def fetch_structure_from_alphafold(uniprot_id, save_dir=STRUCTURE_DIR):
    os.makedirs(save_dir, exist_ok=True)
    for version in ["v4", "v3"]:
        url      = f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_{version}.pdb"
        response = safe_request(url, return_json=False)
        if response and response.status_code == 200:
            file_path = os.path.join(save_dir, f"AF_{uniprot_id}.pdb")
            with open(file_path, "wb") as f: f.write(response.content)
            if os.path.getsize(file_path) > 1000:
                print(f"✅ AlphaFold structure: {uniprot_id}")
                return {"source": "AlphaFold", "pdb_id": f"AF-{uniprot_id}", "file_path": file_path}
    print("⚠️ AlphaFold not available")
    return None


def download_structure(uniprot_id):
    print(f"\n🔍 Checking RCSB for {uniprot_id}...")
    best_pdb = fetch_best_pdb(uniprot_id)
    if best_pdb:
        os.makedirs(STRUCTURE_DIR, exist_ok=True)
        pdb_url   = f"https://files.rcsb.org/download/{best_pdb}.pdb"
        file_path = os.path.join(STRUCTURE_DIR, f"{best_pdb}.pdb")
        response  = safe_request(pdb_url, return_json=False)
        if response and response.status_code == 200:
            with open(file_path, "wb") as f: f.write(response.content)
            if os.path.getsize(file_path) > 1000:
                print(f"✅ Best PDB: {best_pdb}")
                return {"source": "RCSB", "pdb_id": best_pdb, "file_path": file_path}
    print("⚠️ Using AlphaFold fallback")
    return fetch_structure_from_alphafold(uniprot_id)


print("✅ Structure download functions loaded")

✅ Structure download functions loaded


In [7]:
# ══════════════════════════════════════════════════════════════
# PDB CLEANING — TWO STAGE PROCESS
# ══════════════════════════════════════════════════════════════
#
# STAGE 1: clean_pdb()
#   Input:  structures/{pdb_id}.pdb          (raw downloaded file)
#   Output: structures/{pdb_id}_clean.pdb    (removes HETATM/water)
#   Used for: P2Rank pocket detection
#
# STAGE 2: clean_pdb_final() / ensure_final_clean()
#   Input:  structures/{pdb_id}_clean.pdb    (stage 1 output)
#   Output: cleaned_structures/{pdb_id}_clean_final.pdb
#   Used for: Meeko receptor preparation (must be fully clean)
#
# ══════════════════════════════════════════════════════════════

class CleanProtein(Select):
    """Stage 1 clean — removes HETATM and water residues only."""
    def accept_residue(self, residue):
        return residue.id[0] == " "   # keep only ATOM records, not HETATM


def clean_pdb(input_pdb, output_pdb=None):
    """
    Stage 1 cleaning — removes water and heteroatoms.
    Saves to structures/ folder.
    Output used by P2Rank for pocket detection.
    """
    if not os.path.exists(input_pdb) or os.path.getsize(input_pdb) < 1000:
        print(f"❌ Invalid input PDB: {input_pdb}"); return None
    if output_pdb is None:
        base = os.path.basename(input_pdb).replace(".pdb", "_clean.pdb")
        output_pdb = os.path.join(STRUCTURE_DIR, base)   # → structures/
    try:
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure("protein", input_pdb)
        io = PDBIO(); io.set_structure(structure); io.save(output_pdb, CleanProtein())
        print(f"✅ Stage-1 clean saved: {output_pdb}")
        return output_pdb
    except Exception as e:
        print(f"❌ Error in clean_pdb: {e}"); return None


class FinalClean(Select):
    """
    Stage 2 clean — additionally removes:
    - Disordered atoms
    - Atoms with empty/unknown element
    - Problem residues (residue numbers 240-244 which cause Meeko issues)
    """
    def accept_atom(self, atom):
        element = atom.element.strip()
        if element == "" or element == "X": return False
        if atom.is_disordered(): return False
        return True

    def accept_residue(self, residue):
        if residue.id[0] != " ": return False          # remove HETATM
        if residue.id[1] in range(240, 245): return False  # remove problem residues
        return True


def clean_pdb_final(input_pdb, output_dir="cleaned_structures"):
    """
    Stage 2 cleaning — deep clean for Meeko receptor preparation.
    Saves to cleaned_structures/ folder.
    Input:  structures/{pdb_id}_clean.pdb  (stage 1 output)
    Output: cleaned_structures/{pdb_id}_clean_final.pdb
    """
    os.makedirs(output_dir, exist_ok=True)
    pdb_name   = os.path.basename(input_pdb).replace(".pdb", "")
    output_pdb = os.path.join(output_dir, f"{pdb_name}_final.pdb")
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", input_pdb)
    io = PDBIO(); io.set_structure(structure); io.save(output_pdb, FinalClean())
    print(f"✅ Stage-2 clean saved: {output_pdb}")
    return output_pdb


def ensure_final_clean(pdb_file):
    """
    Safe wrapper around clean_pdb_final.
    Checks if cleaned_structures/{base}_clean_final.pdb already exists.
    If yes → reuses it (avoids re-cleaning on re-runs).
    If no  → runs FinalClean and saves it.

    Input:  structures/{pdb_id}_clean.pdb   (stage 1 output from clean_pdb)
    Output: cleaned_structures/{pdb_id}_clean_final.pdb
    """
    pdb_name  = os.path.basename(pdb_file).replace(".pdb", "")
    base_name = pdb_name.replace("_clean", "").replace("_final", "")
    output_dir = "cleaned_structures"
    os.makedirs(output_dir, exist_ok=True)
    expected = os.path.join(output_dir, f"{base_name}_clean_final.pdb")

    if os.path.exists(expected) and os.path.getsize(expected) > 1000:
        print(f"✅ Reusing existing stage-2 clean: {expected}")
        return expected

    if not os.path.exists(pdb_file):
        raise FileNotFoundError(
            f"Stage-1 cleaned PDB not found: {pdb_file}\n"
            f"Run clean_pdb() first on the raw downloaded structure."
        )

    print(f"   Running stage-2 clean: {pdb_file} → {expected}")
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_file)
    io = PDBIO()
    io.set_structure(structure)
    io.save(expected, FinalClean())
    print(f"✅ Stage-2 clean saved: {expected}")
    return expected


print("✅ PDB cleaning functions loaded")
print("   clean_pdb()          → structures/{pdb_id}_clean.pdb       (for P2Rank)")
print("   clean_pdb_final()    → cleaned_structures/{pdb_id}_final.pdb")
print("   ensure_final_clean() → cleaned_structures/{pdb_id}_clean_final.pdb  (for Meeko)")


✅ PDB cleaning functions loaded
   clean_pdb()          → structures/{pdb_id}_clean.pdb       (for P2Rank)
   clean_pdb_final()    → cleaned_structures/{pdb_id}_final.pdb
   ensure_final_clean() → cleaned_structures/{pdb_id}_clean_final.pdb  (for Meeko)


In [8]:
# ══════════════════════════════════════════════════════════════
# P2RANK POCKET DETECTION (UNCHANGED)
# ══════════════════════════════════════════════════════════════

def run_p2rank(pdb_file):
    pdb_file   = os.path.abspath(pdb_file)
    output_dir = os.path.abspath(P2RANK_DIR)
    os.makedirs(output_dir, exist_ok=True)
    cmd = [P2RANK_PATH, "predict", "-f", pdb_file, "-o", output_dir]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True)
        print("STDOUT:\n", result.stdout)
        print("STDERR:\n", result.stderr)
        if result.returncode != 0: print("❌ P2Rank failed"); return None
        print("✅ P2Rank completed")
        return output_dir
    except Exception as e:
        print(f"❌ Error running P2Rank: {e}"); return None


def parse_p2rank_output(output_dir, pdb_file):
    if output_dir is None: print("❌ No P2Rank output directory"); return None, None
    pdb_name  = os.path.basename(pdb_file)
    pred_file = os.path.join(output_dir, f"{pdb_name}_predictions.csv")
    if not os.path.exists(pred_file):
        for f in os.listdir(output_dir):
            if f.endswith("_predictions.csv") and pdb_name in f:
                pred_file = os.path.join(output_dir, f); break
    if not os.path.exists(pred_file):
        print("❌ Predictions file not found")
        print("📂 Available:", os.listdir(output_dir))
        return None, None
    print(f"✅ Using prediction file: {pred_file}")
    df = pd.read_csv(pred_file); df.columns = df.columns.str.strip()
    if df.empty: return None, None
    df = df.sort_values("probability", ascending=False)
    pockets = [{"name": str(row["name"]).strip(), "center_x": float(row["center_x"]),
                "center_y": float(row["center_y"]), "center_z": float(row["center_z"]),
                "score": float(row["score"]), "probability": float(row["probability"])}
               for _, row in df.iterrows()]
    return pockets, df


def save_all_pockets(df, pdb_id):
    pockets_dir = os.path.join(BASE_DIR, "pockets")
    os.makedirs(pockets_dir, exist_ok=True)
    raw_file = os.path.join(pockets_dir, f"{pdb_id}_all_pockets.csv")
    df.to_csv(raw_file, index=False)
    print(f"✅ Saved ALL pockets: {raw_file}")
    return raw_file


def save_clean_pockets(df, pdb_id, top_k=None, prob_threshold=None):
    pockets_dir = os.path.join(BASE_DIR, "pockets")
    os.makedirs(pockets_dir, exist_ok=True)
    df = df.copy(); df.columns = df.columns.str.strip(); df["name"] = df["name"].astype(str).str.strip()
    df = df.sort_values("probability", ascending=False)
    if prob_threshold is not None: df = df[df["probability"] >= prob_threshold]
    if top_k is not None: df = df.head(top_k)
    clean_file = os.path.join(pockets_dir, f"{pdb_id}_clean_pockets.csv")
    df.to_csv(clean_file, index=False)
    print(f"✅ Saved CLEAN pockets: {clean_file}")
    return df


def process_pockets(df, pdb_id, top_k=None, prob_threshold=None):
    if df is None or df.empty: print("❌ No pocket data"); return None
    save_all_pockets(df, pdb_id)
    clean_df = save_clean_pockets(df, pdb_id, top_k=top_k, prob_threshold=prob_threshold)
    print("\n🔹 Selected pockets:")
    print(clean_df[["name", "probability"]])
    return clean_df


# NOTE: load_selected_pockets() and load_clean_pockets() are defined
# in Cell 13 with robust multi-pattern file search.
# Do not redefine them here.


print("✅ P2Rank functions loaded")

✅ P2Rank functions loaded


In [9]:
# ══════════════════════════════════════════════════════════════
# LIGAND RETRIEVAL (UNCHANGED)
# ══════════════════════════════════════════════════════════════

def get_pdb_crystal_ligands(pdb_id):
    data = safe_request(f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}")
    if data is None: return []
    ligands = []
    try: entities = data["rcsb_entry_container_identifiers"]["non_polymer_entity_ids"]
    except: return []
    for entity_id in entities:
        ligand_data = safe_request(f"https://data.rcsb.org/rest/v1/core/nonpolymer_entity/{pdb_id}/{entity_id}")
        if ligand_data is None: continue
        try: ligands.append(ligand_data["pdbx_entity_nonpoly"]["comp_id"])
        except: continue
    return list(set(ligands))


def pdb_ligand_to_smiles(comp_id):
    data = safe_request(f"https://data.rcsb.org/rest/v1/core/chemcomp/{comp_id}")
    return data.get("rcsb_chem_comp_descriptor", {}).get("smiles") if data else None


def get_correct_chembl_target(uniprot_id):
    """
    Find the CORRECT ChEMBL target for a specific UniProt ID.

    ROOT CAUSE OF BUG:
      ChEMBL API returns multiple targets for a UniProt search.
      Blindly picking targets[0] always returns the same first entry
      (e.g. CHEMBL2074 for any protein that shares a complex or pathway).

    FIX:
      1. Search ChEMBL using the exact UniProt accession
      2. From results, prefer target_type == "SINGLE PROTEIN"
      3. Among those, find the one whose target_components contain
         the exact UniProt accession we queried
      4. Verify by checking component xrefs for our UniProt ID
    """
    url  = (f"https://www.ebi.ac.uk/chembl/api/data/target"
            f"?target_components.accession={uniprot_id}&format=json")
    data = safe_request(url)

    if not data or not data.get("targets"):
        print(f"   ❌ No ChEMBL target found for UniProt: {uniprot_id}")
        return None

    targets = data["targets"]
    print(f"   ChEMBL returned {len(targets)} target(s) for {uniprot_id}:")
    for t in targets:
        print(f"     - {t['target_chembl_id']} | {t.get('target_type','?')} | {t.get('pref_name','?')[:60]}")

    # ── Step 1: filter to SINGLE PROTEIN targets only ──
    single_protein_targets = [
        t for t in targets
        if t.get("target_type", "").upper() == "SINGLE PROTEIN"
    ]

    search_pool = single_protein_targets if single_protein_targets else targets

    # ── Step 2: among candidates, find exact UniProt match in components ──
    for t in search_pool:
        components = t.get("target_components", [])
        for comp in components:
            # Each component has xrefs — check for our exact UniProt accession
            xrefs = comp.get("target_component_xrefs", [])
            for xref in xrefs:
                db  = xref.get("xref_src_db", "").upper()
                acc = xref.get("xref_id", "")
                if db in ("UNIPROT", "UNIPROTKB") and acc == uniprot_id:
                    print(f"   ✅ Exact UniProt match: {t['target_chembl_id']} "
                          f"| {t.get('pref_name','?')[:60]}")
                    return t["target_chembl_id"]

    # ── Step 3: fallback — first single protein ──
    if single_protein_targets:
        chosen = single_protein_targets[0]["target_chembl_id"]
        print(f"   ⚠️  No exact xref match — using first SINGLE PROTEIN: {chosen}")
        return chosen

    # ── Step 4: last resort — first result ──
    chosen = targets[0]["target_chembl_id"]
    print(f"   ⚠️  No SINGLE PROTEIN found — using first result: {chosen}")
    return chosen


def get_chembl_ligands(uniprot_id, max_ligands=500):
    """
    Retrieve ligands from ChEMBL for a specific UniProt ID.
    Uses get_correct_chembl_target() to pick the right target —
    not blindly targets[0] which causes same results for different proteins.
    """
    chembl_target = get_correct_chembl_target(uniprot_id)
    if chembl_target is None:
        return []

    print(f"   Fetching activities for {chembl_target} (UniProt: {uniprot_id})...")
    ligands, offset, limit = [], 0, 100

    while len(ligands) < max_ligands:
        act_url = (f"https://www.ebi.ac.uk/chembl/api/data/activity?"
                   f"target_chembl_id={chembl_target}"
                   f"&limit={limit}&offset={offset}&format=json")
        data = safe_request(act_url)
        if not data or not data.get("activities"):
            break
        for act in data["activities"]:
            chembl_id = act.get("molecule_chembl_id")
            if not chembl_id:
                continue
            mol_data = safe_request(
                f"https://www.ebi.ac.uk/chembl/api/data/molecule/{chembl_id}.json"
            )
            if mol_data is None:
                continue
            smiles = mol_data.get("molecule_structures", {}).get("canonical_smiles")
            if smiles:
                ligands.append({"chembl_id": chembl_id, "smiles": smiles})
        offset += limit
        if len(data["activities"]) < limit:
            break

    print(f"   Retrieved {len(ligands)} ligands for {uniprot_id} via {chembl_target}")
    return ligands[:max_ligands]


def canonicalize_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol, canonical=True) if mol else None


def remove_duplicate_ligands(ligands):
    unique = {}
    for lig in ligands:
        canon = canonicalize_smiles(lig["smiles"])
        if canon and canon not in unique: lig["smiles"] = canon; unique[canon] = lig
    return list(unique.values())


def compute_morgan_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048) if mol else None


def tanimoto_similarity(fp1, fp2):
    if fp1 is None or fp2 is None: return 0.0
    return DataStructs.TanimotoSimilarity(fp1, fp2)


def filter_drug_like(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return False

    # Reject salts/mixtures — keep only single-fragment molecules
    # Multi-fragment SMILES crash Meeko during receptor preparation
    frags = Chem.GetMolFrags(mol)
    if len(frags) > 1:
        # Keep largest fragment for property calculation
        mols  = Chem.GetMolFrags(mol, asMols=True)
        mol   = max(mols, key=lambda m: m.GetNumHeavyAtoms())

    return (200 <= Descriptors.MolWt(mol) <= 550 and
            -1  <= Descriptors.MolLogP(mol) <= 5 and
            Descriptors.NumHDonors(mol)     <= 5 and
            Descriptors.NumHAcceptors(mol)  <= 10 and
            Descriptors.NumRotatableBonds(mol) <= 10)

def score_ligands(reference_smiles, ligand_list, top_k=100, similarity_threshold=0.2):
    if not reference_smiles: return ligand_list[:top_k]
    ref_fps = [fp for sm in reference_smiles if (fp := compute_morgan_fp(sm)) is not None]
    if not ref_fps: return ligand_list[:top_k]
    ranked = []
    for lig in ligand_list:
        lig_fp = compute_morgan_fp(lig["smiles"])
        if lig_fp is None: continue
        sims    = [tanimoto_similarity(lig_fp, ref_fp) for ref_fp in ref_fps]
        max_sim = max(sims)
        if max_sim >= similarity_threshold:
            ranked.append({**lig, "similarity_score": round(max_sim, 4),
                           "mean_similarity": round(sum(sims)/len(sims), 4)})
    ranked.sort(key=lambda x: x["similarity_score"], reverse=True)
    return ranked[:top_k]


def select_diverse_ligands(ligands, n=50):
    selected, fps = [], []
    for lig in ligands:
        fp = compute_morgan_fp(lig["smiles"])
        if fp is None: continue
        if not fps or max(DataStructs.TanimotoSimilarity(fp, f) for f in fps) < 0.7:
            selected.append(lig); fps.append(fp)
        if len(selected) >= n: break
    return selected


def retrieve_ligands(uniprot_id, pdb_id):
    print("\n🔍 Retrieving crystal ligands from PDB")
    crystal_ids    = get_pdb_crystal_ligands(pdb_id)
    crystal_smiles = [sm for cid in crystal_ids if (sm := pdb_ligand_to_smiles(cid))]
    crystal_smiles = list(set(crystal_smiles))
    print("Crystal ligands:", len(crystal_smiles))
    print("\n🔍 Retrieving ChEMBL ligands")
    chembl_ligands = [lig for lig in get_chembl_ligands(uniprot_id) if filter_drug_like(lig["smiles"])]
    chembl_ligands = remove_duplicate_ligands(chembl_ligands)
    print("ChEMBL ligands:", len(chembl_ligands))
    if crystal_smiles:
        print("✅ Using similarity scoring")
        return score_ligands(crystal_smiles, chembl_ligands, top_k=100, similarity_threshold=0.2)
    else:
        print("⚠️ No crystal ligand → diverse selection")
        return select_diverse_ligands(chembl_ligands, n=50)


def save_ligands_with_properties(ligands, pdb_id, uniprot_id):
    rows = []
    for lig in ligands:
        mol = Chem.MolFromSmiles(lig.get("smiles", ""))
        if mol is None: continue
        rows.append({"Target": uniprot_id, "PDB_ID": pdb_id, "ChEMBL_ID": lig.get("chembl_id", "NA"),
                     "SMILES": lig["smiles"], "MolWt": round(Descriptors.MolWt(mol), 2),
                     "LogP": round(Descriptors.MolLogP(mol), 2),
                     "HBD": Descriptors.NumHDonors(mol), "HBA": Descriptors.NumHAcceptors(mol)})
    df = pd.DataFrame(rows)
    os.makedirs("ligand_outputs", exist_ok=True)
    file_path = f"ligand_outputs/ligands_{uniprot_id}.csv"
    df.to_csv(file_path, index=False)
    print("✅ Ligand CSV saved:", file_path)
    return df


def load_ligands_from_csv(uniprot_id):
    """
    Load ligands from saved CSV.
    Handles both uppercase (ChEMBL_ID, SMILES) and mixed-case column names.
    """
    file_path = f"ligand_outputs/ligands_{uniprot_id}.csv"
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ File not found: {file_path}")
    df = pd.read_csv(file_path)
    # Normalize column names to uppercase to avoid SMILES vs smiles mismatch
    df.columns = df.columns.str.strip().str.upper()
    ligands = []
    for _, row in df.iterrows():
        ligands.append({
            "chembl_id": row.get("CHEMBL_ID", row.get("CHEMBL_ID", "NA")),
            "smiles":    row["SMILES"],
            "MolWt":     row.get("MOLWT"),
            "LogP":      row.get("LOGP"),
            "HBD":       row.get("HBD"),
            "HBA":       row.get("HBA")
        })
    print(f"✅ Loaded {len(ligands)} ligands from {file_path}")
    return ligands


print("✅ Ligand retrieval functions loaded")

✅ Ligand retrieval functions loaded


In [10]:
# ══════════════════════════════════════════════════════════════
# RECEPTOR PREP & DOCKING — Meeko + Vina
# ══════════════════════════════════════════════════════════════

# ── Helper: load pockets CSV for a given pdb_id ──
def load_selected_pockets(pdb_id):
    """
    Load clean pockets CSV.
    Tries multiple naming patterns to avoid FileNotFoundError.
    pdb_id can be "1V54" or "1V54_clean" — both handled.
    """
    # Strip _clean suffix to get base pdb_id
    base_id = pdb_id.replace("_clean", "").replace("_final", "")

    candidates = [
        f"pockets/{base_id}_clean_clean_pockets.csv",
        f"pockets/{base_id}_pockets.csv",
        f"pockets/{pdb_id}_clean_pockets.csv",
        f"pockets/{pdb_id}_pockets.csv",
        f"pockets/{base_id}.pdb_clean_pockets.csv",
    ]
    file = None
    for c in candidates:
        if os.path.exists(c):
            file = c
            break
    if file is None:
        available = os.listdir("pockets/") if os.path.exists("pockets") else "pockets/ dir missing"
        raise FileNotFoundError(
            f"Pockets CSV not found for {pdb_id}.\n"
            f"Tried:\n" + "\n".join(f"  {c}" for c in candidates) +
            f"\nAvailable in pockets/: {available}"
        )
    df = pd.read_csv(file)
    df.columns = df.columns.str.strip()
    df["name"] = df["name"].str.strip()
    print(f"✅ Loaded {len(df)} pockets from {file}")
    return df


# ── Load clean pockets using the cleaned pdb filepath ──
def load_clean_pockets(cleaned_pdb_file):
    """
    Same as load_selected_pockets but accepts the full cleaned pdb filepath.
    Extracts pdb_id from filename automatically.
    cleaned_pdb_file example: "structures/1V54_clean.pdb"
    """
    pdb_name = os.path.basename(cleaned_pdb_file).replace(".pdb", "")
    return load_selected_pockets(pdb_name)


# ── Ensure PDB is cleaned before receptor prep ──
def ensure_final_clean(pdb_file):
    """
    Checks if a _final.pdb already exists for this PDB.
    If yes, returns the existing path — avoids re-cleaning.
    If no, runs clean_pdb_final() and returns the new path.
    """
    pdb_name   = os.path.basename(pdb_file).replace(".pdb", "")
    # Remove any existing _clean/_final suffixes to get base name
    base_name  = pdb_name.replace("_clean", "").replace("_final", "")
    output_dir = "cleaned_structures"
    os.makedirs(output_dir, exist_ok=True)
    expected   = os.path.join(output_dir, f"{base_name}_clean_final.pdb")

    if os.path.exists(expected) and os.path.getsize(expected) > 1000:
        print(f"✅ Reusing existing cleaned file: {expected}")
        return expected

    # Run FinalClean on whatever pdb_file we have
    print(f"   Cleaning: {pdb_file} → {expected}")
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_file)
    io = PDBIO()
    io.set_structure(structure)
    io.save(expected, FinalClean())
    print(f"✅ Final clean saved: {expected}")
    return expected


# ── Prepare receptor for one pocket via Meeko ──
def prepare_receptor(pdb_file, cx, cy, cz, pocket_id, output_dir="receptors", pdb_id=""):
    """
    Runs Meeko mk_prepare_receptor for one pocket.
    pdb_file MUST be the final-cleaned .pdb (no HETATM, no bad atoms).
    Organises outputs under receptors/{pdb_id}/ folder.
    """
    import sys

    # Verify input file exists and is valid
    if not os.path.exists(pdb_file):
        raise FileNotFoundError(f"Receptor PDB not found: {pdb_file}\nRun ensure_final_clean() first.")
    if os.path.getsize(pdb_file) < 1000:
        raise ValueError(f"Receptor PDB too small (possibly empty): {pdb_file}")

    # Organise under receptors/{pdb_id}/
    if pdb_id:
        output_dir = os.path.join(output_dir, pdb_id)
    os.makedirs(output_dir, exist_ok=True)

    pdb_file      = os.path.abspath(pdb_file)
    pdb_name      = os.path.basename(pdb_file).replace(".pdb", "")
    output_prefix = os.path.join(output_dir, f"{pdb_name}_pocket{pocket_id}")

    cmd = [
        sys.executable, "-m", "meeko.cli.mk_prepare_receptor",
        "--read_pdb", pdb_file,
        "-o", output_prefix,
        "--box_center", str(cx), str(cy), str(cz),
        "--box_size", "25", "25", "25",
        "--allow_bad_res",
        "-p", "-j"
    ]
    print(f"\n📍 Preparing receptor for pocket{pocket_id}")
    print(f"   Input PDB:  {pdb_file}")
    print(f"   Output:     {output_prefix}.*")
    print(f"   Box center: ({cx}, {cy}, {cz})")

    result = subprocess.run(cmd, capture_output=True, text=True)
    print("\n   STDOUT:\n", result.stdout)
    if result.stderr.strip():
        print("\n   STDERR:\n", result.stderr)
    if result.returncode != 0:
        raise Exception(
            f"Meeko failed for pocket{pocket_id}:\n{result.stderr}"
        )

    outputs = {
        "pdbqt":   f"{output_prefix}.pdbqt",
        "json":    f"{output_prefix}.json",
        "box_txt": f"{output_prefix}.box.txt",
        "box_pdb": f"{output_prefix}.box.pdb"
    }
    for key, fpath in outputs.items():
        if os.path.exists(fpath):
            print(f"   ✅ Generated: {fpath}")
        else:
            print(f"   ⚠️  Missing:   {fpath}")
    return outputs


# ── Process protein: clean → load pockets → prepare all receptors ──
def process_protein_with_pockets(pdb_path, pdb_id=""):
    """
    Full receptor preparation pipeline:
    1. ensure_final_clean()  — clean PDB if not already done
    2. load_selected_pockets() — load pocket coordinates
    3. prepare_receptor()    — run Meeko for each pocket

    pdb_path: path to raw or first-cleaned PDB
              e.g. "structures/1V54.pdb" or "structures/1V54_clean.pdb"
    pdb_id:   used to name output folders (e.g. "1V54")
    """
    # Derive pdb_id from filename if not provided
    if not pdb_id:
        pdb_id = os.path.basename(pdb_path).replace(".pdb","").replace("_clean","").replace("_final","")

    print(f"\n🔬 Processing: {pdb_id}")

    # Step 1: Ensure final clean exists (safe — skips if already done)
    final_pdb = ensure_final_clean(pdb_path)

    # Step 2: Load selected pockets — try base pdb_id
    try:
        pockets_df = load_selected_pockets(pdb_id)
    except FileNotFoundError:
        # Fallback: try cleaned filename stem
        stem = os.path.basename(pdb_path).replace(".pdb","")
        pockets_df = load_selected_pockets(stem)

    # Step 3: Prepare receptor for each pocket
    receptors = []
    for i, row in pockets_df.iterrows():
        pocket_name = row["name"]
        cx = float(row["center_x"])
        cy = float(row["center_y"])
        cz = float(row["center_z"])
        pocket_id = i + 1   # 1-indexed

        receptor_output = prepare_receptor(
            pdb_file   = final_pdb,    # always use the final-cleaned file
            cx         = cx,
            cy         = cy,
            cz         = cz,
            pocket_id  = pocket_id,
            output_dir = "receptors",
            pdb_id     = pdb_id        # organise under receptors/{pdb_id}/
        )
        receptors.append({
            "pocket":    pocket_name,
            "pocket_id": pocket_id,
            "center":    (cx, cy, cz),
            "files":     receptor_output
        })

    print(f"\n✅ All pockets processed for {pdb_id}")
    print(f"   Receptors prepared: {len(receptors)}")
    for r in receptors:
        print(f"   - {r['pocket']} → {r['files']['pdbqt']}")
    return receptors


# ── Get receptor .pdbqt file path for a pocket ──
def get_receptor_file(pdb_file, pocket_name, receptor_dir="receptors", pdb_id=""):
    """
    Locate the receptor .pdbqt file for a given pocket.
    Searches inside receptors/{pdb_id}/ first, then receptors/ as fallback.
    Never raises FileNotFoundError — shows clear diagnostics instead.
    """
    # Build search directories
    search_dirs = []
    if pdb_id:
        search_dirs.append(os.path.join(receptor_dir, pdb_id))
    search_dirs.append(receptor_dir)

    candidates = []
    for sdir in search_dirs:
        if not os.path.exists(sdir):
            continue
        for fname in os.listdir(sdir):
            if not fname.endswith(".pdbqt"):
                continue
            # Match by pocket name (e.g. "pocket1") in filename
            if pocket_name in fname:
                candidates.append(os.path.join(sdir, fname))

    if candidates:
        # Most specific match first
        for c in candidates:
            if pocket_name in os.path.basename(c):
                return c
        return candidates[0]

    # Not found — provide clear diagnostic
    all_found = []
    for sdir in search_dirs:
        if os.path.exists(sdir):
            all_found += [os.path.join(sdir, f) for f in os.listdir(sdir) if f.endswith(".pdbqt")]

    raise FileNotFoundError(
        f"Receptor .pdbqt not found for pocket: {pocket_name}\n"
        f"Searched in: {search_dirs}\n"
        f"All .pdbqt files found: {all_found if all_found else 'none'}\n"
        f"Fix: run process_protein_with_pockets() first to generate receptor files."
    )


# ── Prepare one ligand SMILES → .pdbqt ──
def prepare_ligand(smiles, name, folder):
    os.makedirs(folder, exist_ok=True)

    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None

    # Fix: keep only the largest fragment (removes salts/counterions)
    # e.g. "CC(=O)O.CCN" → keeps "CCN" if it has more atoms
    frags = Chem.GetMolFrags(mol, asMols=True)
    if len(frags) > 1:
        mol = max(frags, key=lambda m: m.GetNumHeavyAtoms())

    mol = Chem.AddHs(mol)
    if AllChem.EmbedMolecule(mol) != 0: return None
    AllChem.UFFOptimizeMolecule(mol)

    try:
        preparator   = MoleculePreparation()
        setups       = preparator.prepare(mol)
        writer       = PDBQTWriterLegacy()
        pdbqt_string = writer.write_string(setups[0])[0]
    except Exception as e:
        print(f"   ⚠️ Meeko failed for {name}: {e}")
        return None

    ligand_file = os.path.join(folder, f"{name}.pdbqt")
    with open(ligand_file, "w") as f:
        f.write(pdbqt_string)
    return ligand_file


# ── Run Vina for one ligand ──
def run_vina(ligand_file, receptor_file, cx, cy, cz, pocket_name, pdb_id=""):
    folder = os.path.join("docking_results", pdb_id, pocket_name) if pdb_id else os.path.join("docking_results", pocket_name)
    os.makedirs(folder, exist_ok=True)
    ligand_name = os.path.basename(ligand_file).replace(".pdbqt", "")
    out_file    = os.path.join(folder, f"{ligand_name}_out.pdbqt")
    cmd = [
        VINA,
        "--receptor", receptor_file,
        "--ligand",   ligand_file,
        "--center_x", str(cx), "--center_y", str(cy), "--center_z", str(cz),
        "--size_x", "25", "--size_y", "25", "--size_z", "25",
        "--exhaustiveness", "8",
        "--out", out_file
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print("\nVINA OUTPUT:\n", result.stdout)
    return result.stdout


def extract_vina_score(output):
    for line in output.split("\n"):
        if line.strip().startswith("1 "):
            try: return float(line.split()[1])
            except: return None
    return None


# ── Dock all ligands against one pocket ──
def dock_many_ligands(ligands, pdb_file, pocket_name, cx, cy, cz, pdb_id=""):
    results       = []
    receptor_file = get_receptor_file(pdb_file, pocket_name, pdb_id=pdb_id)
    folder        = os.path.join("docking_results", pdb_id, pocket_name) if pdb_id else os.path.join("docking_results", pocket_name)
    os.makedirs(folder, exist_ok=True)
    print(f"   Receptor:      {receptor_file}")
    print(f"   Output folder: {folder}")
    for i, lig in enumerate(ligands):
        ligand_file = prepare_ligand(lig["smiles"], f"lig_{i}", folder)
        if ligand_file is None: continue
        output = run_vina(ligand_file, receptor_file, cx, cy, cz, pocket_name, pdb_id=pdb_id)
        score  = extract_vina_score(output)
        results.append({**lig, "docking_score": score, "pocket": pocket_name})
    return results


def save_all_results(all_results, pdb_id):
    os.makedirs("final_results", exist_ok=True)
    df = pd.DataFrame(all_results)
    file_path = f"final_results/{pdb_id}_all_docking_results.csv"
    df.to_csv(file_path, index=False)
    print(f"✅ Saved ALL results: {file_path}")
    return df


def save_ranked_results(df, pdb_id):
    os.makedirs("final_results", exist_ok=True)
    df = df[df["docking_score"].notnull()].sort_values("docking_score")
    file_path = f"final_results/{pdb_id}_ranked_results.csv"
    df.to_csv(file_path, index=False)
    print(f"✅ Saved ranked results: {file_path}")
    return df


def save_best_per_pocket(df, pdb_id):
    os.makedirs("final_results", exist_ok=True)
    df = df[df["docking_score"].notnull()]
    best_list = []
    for pocket in df["pocket"].unique():
        best = df[df["pocket"] == pocket].sort_values("docking_score").iloc[0]
        best_list.append(best)
    best_df = pd.DataFrame(best_list)
    file_path = f"final_results/{pdb_id}_best_per_pocket.csv"
    best_df.to_csv(file_path, index=False)
    print(f"✅ Saved best per pocket: {file_path}")
    return best_df


print("✅ Receptor prep & docking functions loaded")
print("   ensure_final_clean()           — clean PDB safely, skip if done")
print("   load_selected_pockets(pdb_id)  — reads pockets CSV, tries multiple names")
print("   prepare_receptor()             — Meeko, saves to receptors/{pdb_id}/")
print("   process_protein_with_pockets() — full receptor prep pipeline")
print("   get_receptor_file()            — finds .pdbqt, clear error if missing")
print("   dock_many_ligands()            — saves to docking_results/{pdb_id}/{pocket}/")


✅ Receptor prep & docking functions loaded
   ensure_final_clean()           — clean PDB safely, skip if done
   load_selected_pockets(pdb_id)  — reads pockets CSV, tries multiple names
   prepare_receptor()             — Meeko, saves to receptors/{pdb_id}/
   process_protein_with_pockets() — full receptor prep pipeline
   get_receptor_file()            — finds .pdbqt, clear error if missing
   dock_many_ligands()            — saves to docking_results/{pdb_id}/{pocket}/


## ─── CELL D: NEW — LLM Pocket–Ligand Binding Explanation ───

In [11]:
# ══════════════════════════════════════════════════════════════
# LLM POCKET EXPLANATION ENGINE  (NEW — Obj-2 extension)
# Uses Ollama HTTP API — safe for meeko_env
# Explains WHY each ligand binds to each specific pocket
# ══════════════════════════════════════════════════════════════

HYDROPHOBIC = {"ALA","VAL","LEU","ILE","MET","PHE","TRP","PRO"}
POLAR       = {"SER","THR","ASN","GLN","TYR","CYS"}
AROMATIC    = {"PHE","TYR","TRP","HIS"}
HBD_SET     = {"SER","THR","TYR","LYS","ARG","HIS"}
HBA_SET     = {"ASP","GLU","ASN","GLN"}


def compute_pocket_features(residues: list) -> Optional[dict]:
    total = len(residues)
    if total == 0: return None
    return {
        "hydrophobic_ratio": sum(r in HYDROPHOBIC for r in residues) / total,
        "polar_ratio":       sum(r in POLAR for r in residues) / total,
        "hbd":               sum(r in HBD_SET for r in residues),
        "hba":               sum(r in HBA_SET for r in residues),
        "aromatic":          sum(r in AROMATIC for r in residues),
        "dominant_residues": [r for r, _ in Counter(residues).most_common(5)]
    }


def compute_ligand_features(smiles: str) -> Optional[dict]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    return {
        "mw":            round(Descriptors.MolWt(mol), 2),
        "logp":          round(Descriptors.MolLogP(mol), 2),
        "hbd":           Descriptors.NumHDonors(mol),
        "hba":           Descriptors.NumHAcceptors(mol),
        "rot":           Descriptors.NumRotatableBonds(mol),
        "aromatic_rings": Chem.rdMolDescriptors.CalcNumAromaticRings(mol)
    }


def get_pocket_residues_from_p2rank(pocket_name: str, pdb_id: str) -> list:
    """Load residue-level pocket data from P2Rank residues CSV."""
    residue_file = os.path.join(P2RANK_DIR, f"{pdb_id}_clean.pdb_residues.csv")
    if not os.path.exists(residue_file):
        for f in os.listdir(P2RANK_DIR):
            if f.endswith("_residues.csv") and pdb_id in f:
                residue_file = os.path.join(P2RANK_DIR, f); break
    if not os.path.exists(residue_file):
        print(f"⚠️ Residues file not found for {pdb_id}"); return []
    res_df = pd.read_csv(residue_file); res_df.columns = res_df.columns.str.strip()
    pocket_rank = int(pocket_name.replace("pocket", ""))
    residues    = res_df[res_df["pocket"] == pocket_rank - 1]["residue_name"].tolist()
    return [r.strip() for r in residues][:60]


def explain_pocket_ligand_binding(
    pocket_name: str, pocket_residues: list, pocket_features: dict,
    ligand_smiles: str, ligand_props: dict, docking_score: float, protein_name: str = ""
) -> str:
    """
    LLM explanation of WHY a ligand binds to a specific pocket.
    Analyzes physicochemical complementarity between pocket and ligand.
    Uses Ollama HTTP API — works in meeko_env without ollama package.
    """
    system = """You are a structural biologist and computational chemist expert in protein–ligand interactions.
Explain binding mechanisms based on physicochemical data only."""

    user = f"""
Protein: {protein_name}
Pocket: {pocket_name}

POCKET PROPERTIES:
- Key residues (first 20): {', '.join(pocket_residues[:20])}
- Dominant residue types: {pocket_features.get('dominant_residues', [])}
- Hydrophobic ratio: {pocket_features['hydrophobic_ratio']:.2f}  (>0.4 = strongly hydrophobic)
- Polar ratio: {pocket_features['polar_ratio']:.2f}
- H-bond donor residues: {pocket_features['hbd']}
- H-bond acceptor residues: {pocket_features['hba']}
- Aromatic residues: {pocket_features['aromatic']}

LIGAND PROPERTIES:
- SMILES: {ligand_smiles}
- MW: {ligand_props['mw']} Da
- LogP: {ligand_props['logp']}
- H-bond donors: {ligand_props['hbd']}
- H-bond acceptors: {ligand_props['hba']}
- Rotatable bonds: {ligand_props['rot']}
- Aromatic rings: {ligand_props['aromatic_rings']}

Docking score: {docking_score} kcal/mol (more negative = stronger binding)

Explain:
1. What interaction types drive binding (hydrophobic, H-bonds, π-stacking, electrostatic)
2. Which pocket residue types match which ligand features
3. How docking score reflects binding strength
4. What makes this pocket druggable or not for this ligand
5. Any drug-likeness concerns (MW/LogP)

Provide a precise, data-driven explanation in 8–10 sentences.
"""
    return call_ollama_http(system, user)


def explain_all_pocket_ligand_pairs(pdb_id: str, uniprot_id: str, protein_name: str = "") -> dict:
    """
    Generate LLM explanations for all best pocket–ligand pairs after docking.
    Call AFTER docking is complete.
    Reads:  final_results/{pdb_id}_best_per_pocket.csv
    Writes: final_results/{pdb_id}_pocket_explanations.json
    """
    best_path = f"final_results/{pdb_id}_best_per_pocket.csv"
    if not os.path.exists(best_path):
        print(f"❌ Docking results not found: {best_path}")
        print("   Run docking first, then call this function.")
        return {}

    best_df      = pd.read_csv(best_path)
    explanations = {}

    for _, row in best_df.iterrows():
        pocket_name   = row.get("pocket", "")
        ligand_smiles = row.get("SMILES", row.get("smiles", ""))
        docking_score = row.get("docking_score", 0)
        chembl_id     = row.get("chembl_id", row.get("ChEMBL_ID", "Unknown"))
        if not ligand_smiles or not pocket_name: continue

        print(f"\n🧠 Explaining: {pocket_name} ← {chembl_id} (score: {docking_score})")

        pocket_residues = get_pocket_residues_from_p2rank(pocket_name, pdb_id)
        pocket_features = compute_pocket_features(pocket_residues) or {
            "hydrophobic_ratio": 0, "polar_ratio": 0, "hbd": 0,
            "hba": 0, "aromatic": 0, "dominant_residues": []}
        ligand_props    = compute_ligand_features(ligand_smiles)
        if ligand_props is None: continue

        explanation = explain_pocket_ligand_binding(
            pocket_name=pocket_name, pocket_residues=pocket_residues,
            pocket_features=pocket_features, ligand_smiles=ligand_smiles,
            ligand_props=ligand_props, docking_score=docking_score,
            protein_name=protein_name or uniprot_id)

        explanations[pocket_name] = {
            "chembl_id": chembl_id, "smiles": ligand_smiles,
            "docking_score": docking_score, "pocket_features": pocket_features,
            "ligand_features": ligand_props, "explanation": explanation}
        print(f"\n📋 [{pocket_name}]\n{explanation}")

    out_path = f"final_results/{pdb_id}_pocket_explanations.json"
    with open(out_path, "w") as f: json.dump(explanations, f, indent=2)
    print(f"\n✅ Pocket explanations saved: {out_path}")
    return explanations


print("✅ LLM Pocket explanation engine loaded")

✅ LLM Pocket explanation engine loaded


## ─── CELL D2: Visualization (removed — will be added later) ───

In [12]:
# Visualization functions removed — will be added in a separate cell later
print("Visualization skipped")

Visualization skipped


## ─── CELL E: LangGraph Agent ───

In [13]:
# ══════════════════════════════════════════════════════════════
# LANGGRAPH STATE for Obj-2
# ══════════════════════════════════════════════════════════════

class StructureState(TypedDict):
    raw_query:     str   # original natural language query if any
    uniprot_id:    str
    protein_name:  str
    pdb_id:        str
    pdb_file:      str
    cleaned_pdb:   str
    pockets:       list
    ligands:       list
    docking_done:  bool
    explanations:  dict
    messages:      Annotated[list, operator.add]


# ─── Node 1: Download & Clean ───
def node_download_structure(state: StructureState) -> StructureState:
    uniprot_id = state["uniprot_id"]
    print(f"\n📥 Downloading structure for {uniprot_id}")
    result = download_structure(uniprot_id)
    if result is None:
        return {**state, "messages": [AIMessage(content=f"❌ No structure found for {uniprot_id}")]}
    pdb_file = result["file_path"]
    pdb_id   = result["pdb_id"]
    cleaned  = clean_pdb(pdb_file)
    print(f"   Source: {result['source']} | PDB: {pdb_id}")
    return {**state, "pdb_id": pdb_id, "pdb_file": pdb_file, "cleaned_pdb": cleaned or pdb_file,
            "messages": [AIMessage(content=f"Structure: {pdb_id} from {result['source']}")]}


# ─── Node 2: Detect Pockets ───
def node_detect_pockets(state: StructureState) -> StructureState:
    cleaned_pdb = state["cleaned_pdb"]
    pdb_id      = state["pdb_id"]
    print(f"\n🔍 Running P2Rank on {os.path.basename(cleaned_pdb)}")
    output_dir = run_p2rank(cleaned_pdb)
    pockets, df = None, None
    if output_dir:
        pockets, df = parse_p2rank_output(output_dir, cleaned_pdb)
    if pockets and df is not None:
        # Use pdb_id (e.g. "1BCD") not cleaned filename (e.g. "1BCD_clean")
        # so file saves as pockets/1BCD_clean_pockets.csv
        # which load_selected_pockets("1BCD") can find correctly
        process_pockets(df, pdb_id, top_k=3, prob_threshold=0.5)
        print(f"   {len(pockets)} pockets detected")
    else:
        pockets = []; print("   ⚠️ No pockets detected")
    return {**state, "pockets": pockets or [],
            "messages": [AIMessage(content=f"{len(pockets or [])} pockets detected")]}

# ─── Node 3: Retrieve Ligands ───
def node_retrieve_ligands(state: StructureState) -> StructureState:
    uniprot_id = state["uniprot_id"]
    pdb_id     = state["pdb_id"]
    print(f"\n💊 Retrieving ligands for {uniprot_id}")
    ligands = retrieve_ligands(uniprot_id, pdb_id)
    if ligands: save_ligands_with_properties(ligands, pdb_id, uniprot_id)
    print(f"   {len(ligands)} ligands retrieved")
    return {**state, "ligands": ligands,
            "messages": [AIMessage(content=f"{len(ligands)} ligands from PDB+ChEMBL")]}


# ─── Node 4: Prepare receptors & Dock ───
def node_dock(state: StructureState) -> StructureState:
    pdb_id      = state["pdb_id"]
    pdb_file    = state["pdb_file"]
    cleaned_pdb = state["cleaned_pdb"]
    uniprot_id  = state["uniprot_id"]
    print(f"\n⚗️ Starting docking for {pdb_id}")
    try:
        # Step 1: Ensure final clean exists (safe — skips if already done)
        final_clean = ensure_final_clean(cleaned_pdb)

        # Step 2: Load pockets using base pdb_id
        clean_df = load_selected_pockets(pdb_id)

        # Step 3: Prepare receptors — organise under receptors/{pdb_id}/
        for i, row in clean_df.iterrows():
            prepare_receptor(
                pdb_file   = final_clean,
                cx         = float(row["center_x"]),
                cy         = float(row["center_y"]),
                cz         = float(row["center_z"]),
                pocket_id  = i + 1,
                output_dir = "receptors",
                pdb_id     = pdb_id
            )

        # Step 4: Load ligands
        ligands = load_ligands_from_csv(uniprot_id)

        # Step 5: Dock all ligands across all pockets
        all_results = []
        for _, pocket in clean_df.iterrows():
            pocket_name = pocket["name"].strip()
            print(f"\n🔬 Docking in {pocket_name}")
            results = dock_many_ligands(
                ligands     = ligands,
                pdb_file    = final_clean,
                pocket_name = pocket_name,
                cx          = float(pocket["center_x"]),
                cy          = float(pocket["center_y"]),
                cz          = float(pocket["center_z"]),
                pdb_id      = pdb_id     # saves to docking_results/{pdb_id}/{pocket}/
            )
            all_results.extend(results)

        df = save_all_results(all_results, pdb_id)
        save_ranked_results(df, pdb_id)
        save_best_per_pocket(df, pdb_id)
        return {**state, "docking_done": True,
                "messages": [AIMessage(content=f"Docking complete: {len(all_results)} runs")]}
    except Exception as e:
        import traceback
        print(f"❌ Docking error: {e}")
        print(traceback.format_exc())
        return {**state, "docking_done": False,
                "messages": [AIMessage(content=f"Docking failed: {e}")]}


# ─── Node 5: LLM Pocket Explanation (NEW) ───
def node_pocket_explanation(state: StructureState) -> StructureState:
    if not state.get("docking_done"):
        print("⚠️ Docking not complete — skipping pocket explanation")
        return {**state, "explanations": {},
                "messages": [AIMessage(content="Pocket explanation skipped (no docking results)")]}
    print(f"\n🧠 Generating pocket binding explanations for {state['pdb_id']}")
    explanations = explain_all_pocket_ligand_pairs(state["pdb_id"], state["uniprot_id"], state.get("protein_name", ""))
    return {**state, "explanations": explanations,
            "messages": [AIMessage(content=f"Generated {len(explanations)} pocket binding explanations")]}




# ─── Node 6: Export Shared ───
def node_export_shared(state: StructureState) -> StructureState:
    pdb_id       = state["pdb_id"]
    explanations = state.get("explanations", {})
    pocket_summary = [{"pocket": p, "chembl_id": d["chembl_id"], "docking_score": d["docking_score"],
                       "explanation_snippet": d["explanation"][:300]} for p, d in explanations.items()]
    shared_payload = {
        "uniprot_id":         state["uniprot_id"],
        "pdb_id":             pdb_id,
        "pockets_detected":   len(state.get("pockets", [])),
        "ligands_retrieved":  len(state.get("ligands", [])),
        "docking_done":       state.get("docking_done", False),
        "pocket_explanations": pocket_summary,
        "explanation_file":   f"final_results/{pdb_id}_pocket_explanations.json"
    }
    with open(STRUCTURE_OUTPUT_FILE, "w") as f: json.dump(shared_payload, f, indent=2)
    print(f"\n✅ shared/structure_output.json written")
    return {**state, "messages": [AIMessage(content=f"Structure analysis complete for {pdb_id}")]}


print("✅ LangGraph nodes defined for Obj-2")

✅ LangGraph nodes defined for Obj-2


In [14]:
# ══════════════════════════════════════════════════════════════
# BUILD LANGGRAPH FOR OBJ-2
# Flow: download → detect_pockets → retrieve_ligands → dock
#       → pocket_explanation → export_shared → END
# ══════════════════════════════════════════════════════════════

struct_builder = StateGraph(StructureState)
struct_builder.add_node("download_structure", node_download_structure)
struct_builder.add_node("detect_pockets",      node_detect_pockets)
struct_builder.add_node("retrieve_ligands",    node_retrieve_ligands)
struct_builder.add_node("dock",                node_dock)
struct_builder.add_node("pocket_explanation",  node_pocket_explanation)
struct_builder.add_node("export_shared",       node_export_shared)

struct_builder.set_entry_point("download_structure")
struct_builder.add_edge("download_structure", "detect_pockets")
struct_builder.add_edge("detect_pockets",      "retrieve_ligands")
struct_builder.add_edge("retrieve_ligands",    "dock")
struct_builder.add_edge("dock",                "pocket_explanation")
struct_builder.add_edge("pocket_explanation",  "export_shared")
struct_builder.add_edge("export_shared",        END)

struct_agent = struct_builder.compile()
print("✅ Obj-2 LangGraph compiled:", list(struct_builder.nodes.keys()))

✅ Obj-2 LangGraph compiled: ['download_structure', 'detect_pockets', 'retrieve_ligands', 'dock', 'pocket_explanation', 'export_shared']


## ─── CELL F: Run Functions ───

In [15]:
def run_structure_agent(uniprot_id: str, protein_name: str = "") -> dict:
    """
    Run the Obj-2 Structure Agent for a given UniProt ID.
    Steps: Download → Clean → P2Rank → ChEMBL+PDB Ligands → Meeko → Vina → LLM Explanation
    """
    print("\n" + "═"*60)
    print(f"  OBJ-2 STRUCTURE AGENT  |  Python 3.10.20 / meeko_env")
    print(f"  UniProt: {uniprot_id}  |  {protein_name}")
    print("═"*60)

    initial: StructureState = {
        "raw_query":    uniprot_id,   # pass through for normalizer
        "uniprot_id":   uniprot_id, "protein_name": protein_name,
        "pdb_id":       "", "pdb_file": "", "cleaned_pdb": "",
        "pockets":      [], "ligands": [], "docking_done": False,
        "explanations": {},
        "messages": [HumanMessage(content=f"Analyse protein {uniprot_id}")]
    }
    final = struct_agent.invoke(initial)
    print("\n" + "═"*60)
    print("  OBJ-2 COMPLETE — shared/structure_output.json written")
    print("═"*60)
    return final


def run_structure_agent_from_kg_output() -> list:
    """
    Read shared/kg_output.json (written by Obj-1) and run the structure agent
    for each druggable target identified by the KG agent.
    This is the integration bridge between Obj-1 and Obj-2.
    """
    if not os.path.exists(KG_INPUT_FILE):
        print(f"❌ {KG_INPUT_FILE} not found.")
        print("   Run obj_1_kg_agent.ipynb first with DISEASE_DRUGGABLE_TARGETS or FULL_PIPELINE.")
        return []

    with open(KG_INPUT_FILE) as f:
        kg_data = json.load(f)

    targets      = kg_data.get("druggable_targets", [])
    disease_name = kg_data.get("disease", {}).get("disease_name", "Unknown disease")

    print(f"\n📂 Reading KG output for: {disease_name}")
    print(f"   Targets from Obj-1: {[t['gene_name'] for t in targets]}")

    results = []
    for t in targets:
        result = run_structure_agent(t["protein_id"], t.get("gene_name", t.get("protein_name", "")))
        results.append(result)
    return results


def run_pocket_explanation_only(pdb_id: str, uniprot_id: str, protein_name: str = "") -> dict:
    """
    Run only the LLM pocket explanation step on existing docking results.
    Use when you already have final_results/{pdb_id}_best_per_pocket.csv
    """
    print(f"\n🧠 Running pocket explanation for {pdb_id} / {uniprot_id}")
    return explain_all_pocket_ligand_pairs(pdb_id, uniprot_id, protein_name)


print("✅ run_structure_agent(), run_structure_agent_from_kg_output(), run_pocket_explanation_only() ready")

# ── Manual visualization on existing results ──
# Use this if you already have docking results and want to regenerate plots
# final = run_structure_agent("P07471", protein_name="Alpha-1-antitrypsin")
# visualize_all(final)


✅ run_structure_agent(), run_structure_agent_from_kg_output(), run_pocket_explanation_only() ready


## ─── CELL G: Example Runs ───

In [32]:
# ── Standalone: your original usage, now through LangGraph ──
run_structure_agent("P07471", protein_name="Alpha-1-antitrypsin")


════════════════════════════════════════════════════════════
  OBJ-2 STRUCTURE AGENT  |  Python 3.10.20 / meeko_env
  UniProt: P07471  |  Alpha-1-antitrypsin
════════════════════════════════════════════════════════════

📥 Downloading structure for P07471

🔍 Checking RCSB for P07471...
   Found 93 PDB entries — selecting best...
   1OCC: method=X-ray, res=2.8, ligands=['14', '15', '16', '17'], score=55
   1OCO: method=X-ray, res=2.8, ligands=['14', '15', '16', '17', '18', '19'], score=55
   1OCR: method=X-ray, res=2.35, ligands=['14', '15', '16', '17', '18'], score=60
   1OCZ: method=X-ray, res=2.9, ligands=['14', '15', '16', '17', '18', '19'], score=55


KeyboardInterrupt: 

In [40]:
run_structure_agent("P00918", protein_name="Human Carbonic Anhydrase II")


════════════════════════════════════════════════════════════
  OBJ-2 STRUCTURE AGENT  |  Python 3.10.20 / meeko_env
  UniProt: P00918  |  Human Carbonic Anhydrase II
════════════════════════════════════════════════════════════

📥 Downloading structure for P00918

🔍 Checking RCSB for P00918...
   Found 1200 PDB entries — selecting best...
   12CA: method=X-ray, res=2.4, ligands=['2', '3'], score=60
   1A42: method=X-ray, res=2.25, ligands=['2', '3', '4'], score=60
   1AM6: method=X-ray, res=2.0, ligands=['2', '3', '4'], score=60
   1AVN: method=X-ray, res=2.0, ligands=['2', '3', '4', '5'], score=60
   1BCD: method=X-ray, res=1.9, ligands=['2', '3'], score=65
   1BIC: method=X-ray, res=1.9, ligands=['2', '3', '4'], score=65
   1BN1: method=X-ray, res=2.1, ligands=['2', '3', '4'], score=60
   1BN3: method=X-ray, res=2.2, ligands=['2', '3', '4'], score=60
   1BN4: method=X-ray, res=2.1, ligands=['2', '3', '4'], score=60
   1BNM: method=X-ray, res=2.6, ligands=['2', '3', '4'], score=55
   

{'raw_query': 'P00918',
 'uniprot_id': 'P00918',
 'protein_name': 'Human Carbonic Anhydrase II',
 'pdb_id': '1BCD',
 'pdb_file': 'C:\\Dissertation\\code\\structures\\1BCD.pdb',
 'cleaned_pdb': 'C:\\Dissertation\\code\\structures\\1BCD_clean.pdb',
 'pockets': [{'name': 'pocket1',
   'center_x': -5.0486,
   'center_y': 2.9595,
   'center_z': 14.4112,
   'score': 24.57,
   'probability': 0.899},
  {'name': 'pocket2',
   'center_x': 4.8974,
   'center_y': -4.2348,
   'center_z': 6.6136,
   'score': 8.07,
   'probability': 0.429}],
 'ligands': [{'chembl_id': 'CHEMBL268177', 'smiles': 'Nc1ccc(S(N)(=O)=O)cc1I'},
  {'chembl_id': 'CHEMBL268439',
   'smiles': 'Nc1ccc(S(=O)(=O)Nc2nnc(S(N)(=O)=O)s2)cc1'},
  {'chembl_id': 'CHEMBL544127',
   'smiles': 'CC(C)CN(C)Cc1cc(C(=O)c2csc(S(N)(=O)=O)c2)ccc1O.Cl'},
  {'chembl_id': 'CHEMBL358290',
   'smiles': 'NS(=O)(=O)c1ccc(N/C(S)=N\\CCc2ccccc2)cc1'},
  {'chembl_id': 'CHEMBL148426',
   'smiles': 'NS(=O)(=O)c1ccc(N/C(S)=N\\CCc2c[nH]cn2)cc1'},
  {'chembl_id': 

In [41]:
run_structure_agent("P43490", protein_name="Nicotinamide phosphoribosyltransferase")


════════════════════════════════════════════════════════════
  OBJ-2 STRUCTURE AGENT  |  Python 3.10.20 / meeko_env
  UniProt: P43490  |  Nicotinamide phosphoribosyltransferase
════════════════════════════════════════════════════════════

📥 Downloading structure for P43490

🔍 Checking RCSB for P43490...
   Found 84 PDB entries — selecting best...
   2E5B: method=X-ray, res=2.0, ligands=[], score=40
   2E5C: method=X-ray, res=2.2, ligands=['2'], score=60
   2E5D: method=X-ray, res=2.0, ligands=['2'], score=60
   2GVG: method=X-ray, res=2.2, ligands=['2', '3'], score=60
   2GVJ: method=X-ray, res=2.1, ligands=['2'], score=60
   3DGR: method=X-ray, res=2.1, ligands=['2'], score=60
   3DHD: method=X-ray, res=2.0, ligands=['2', '3', '4'], score=60
   3DHF: method=X-ray, res=1.8, ligands=['2', '3', '4', '5'], score=65
   3DKJ: method=X-ray, res=2.0, ligands=['2', '3'], score=60
   3DKL: method=X-ray, res=1.89, ligands=['2', '3', '4', '5'], score=65
   4JNM: method=X-ray, res=2.2, ligands=['

{'raw_query': 'P43490',
 'uniprot_id': 'P43490',
 'protein_name': 'Nicotinamide phosphoribosyltransferase',
 'pdb_id': '3DHF',
 'pdb_file': 'C:\\Dissertation\\code\\structures\\3DHF.pdb',
 'cleaned_pdb': 'C:\\Dissertation\\code\\structures\\3DHF_clean.pdb',
 'pockets': [{'name': 'pocket1',
   'center_x': 7.6285,
   'center_y': -0.7184,
   'center_z': 38.9688,
   'score': 77.39,
   'probability': 0.993},
  {'name': 'pocket2',
   'center_x': 11.5943,
   'center_y': 15.5955,
   'center_z': 6.1917,
   'score': 76.73,
   'probability': 0.993},
  {'name': 'pocket3',
   'center_x': 25.4675,
   'center_y': 6.754,
   'center_z': 32.667,
   'score': 4.7,
   'probability': 0.206},
  {'name': 'pocket4',
   'center_x': 27.4149,
   'center_y': 5.7461,
   'center_z': 16.8547,
   'score': 4.32,
   'probability': 0.179},
  {'name': 'pocket5',
   'center_x': 18.4297,
   'center_y': -23.8898,
   'center_z': 24.6334,
   'score': 3.45,
   'probability': 0.125},
  {'name': 'pocket6',
   'center_x': 19.1118,

In [42]:
run_structure_agent("P12821", protein_name="Angiotensin converting enzyme ")


════════════════════════════════════════════════════════════
  OBJ-2 STRUCTURE AGENT  |  Python 3.10.20 / meeko_env
  UniProt: P12821  |  Angiotensin converting enzyme 
════════════════════════════════════════════════════════════

📥 Downloading structure for P12821

🔍 Checking RCSB for P12821...
   Found 97 PDB entries — selecting best...
   1O86: method=X-ray, res=2.0, ligands=['2', '3', '4', '5'], score=60
   1O8A: method=X-ray, res=2.0, ligands=['2', '3', '4', '5', '6'], score=60
   1UZE: method=X-ray, res=1.82, ligands=['2', '3', '4', '5'], score=65
   1UZF: method=X-ray, res=2.0, ligands=['2', '3', '4', '5'], score=60
   2C6F: method=X-ray, res=3.01, ligands=['3', '4', '5', '6', '7'], score=50
   2C6N: method=X-ray, res=3.0, ligands=['3', '4', '5', '6', '7', '8'], score=50
   2IUL: method=X-ray, res=2.01, ligands=['3', '4', '5', '6'], score=60
   2IUX: method=X-ray, res=2.8, ligands=['2', '3', '4', '5', '6'], score=55
   2OC2: method=X-ray, res=2.25, ligands=['2', '3', '4'], scor

{'raw_query': 'P12821',
 'uniprot_id': 'P12821',
 'protein_name': 'Angiotensin converting enzyme ',
 'pdb_id': '1UZE',
 'pdb_file': 'C:\\Dissertation\\code\\structures\\1UZE.pdb',
 'cleaned_pdb': 'C:\\Dissertation\\code\\structures\\1UZE_clean.pdb',
 'pockets': [{'name': 'pocket1',
   'center_x': 42.3415,
   'center_y': 32.6526,
   'center_z': 44.8078,
   'score': 110.98,
   'probability': 0.997},
  {'name': 'pocket2',
   'center_x': 35.9026,
   'center_y': 46.7938,
   'center_z': 60.2426,
   'score': 12.94,
   'probability': 0.668},
  {'name': 'pocket3',
   'center_x': 28.9129,
   'center_y': 45.7531,
   'center_z': 52.3618,
   'score': 5.38,
   'probability': 0.255},
  {'name': 'pocket4',
   'center_x': 59.1834,
   'center_y': 27.6039,
   'center_z': 28.522,
   'score': 4.53,
   'probability': 0.194},
  {'name': 'pocket5',
   'center_x': 36.7436,
   'center_y': 52.6647,
   'center_z': 51.2939,
   'score': 3.09,
   'probability': 0.103},
  {'name': 'pocket6',
   'center_x': 31.9167,
 

In [44]:

run_structure_agent("Q96C23", protein_name="Galactose mutarotase ")


════════════════════════════════════════════════════════════
  OBJ-2 STRUCTURE AGENT  |  Python 3.10.20 / meeko_env
  UniProt: Q96C23  |  Galactose mutarotase 
════════════════════════════════════════════════════════════

📥 Downloading structure for Q96C23

🔍 Checking RCSB for Q96C23...
   Found 2 PDB entries — selecting best...
   1SNZ: method=X-ray, res=2.2, ligands=[], score=40
   1SO0: method=X-ray, res=2.3, ligands=['2'], score=60

   Best PDB selected: 1SO0 (score=60, resolution=2.3, ligands=['2'])
✅ Best PDB: 1SO0
✅ Stage-1 clean saved: C:\Dissertation\code\structures\1SO0_clean.pdb
   Source: RCSB | PDB: 1SO0

🔍 Running P2Rank on 1SO0_clean.pdb
STDOUT:
 P2Rank 2.5.1

DIR: .
DIR2: .
DIR: C:\Dissertation\p2rank_2.5.1\config/./output
DIR2: C:\Dissertation\p2rank_2.5.1\config/./output
predicting pockets for proteins from dataset [1SO0_clean.pdb]
processing [1SO0_clean.pdb] (1/1)
predicting pockets finished in 0 hours 0 minutes 3.846 seconds
results saved to directory [C:\Dissertat

{'raw_query': 'Q96C23',
 'uniprot_id': 'Q96C23',
 'protein_name': 'Galactose mutarotase ',
 'pdb_id': '1SO0',
 'pdb_file': 'C:\\Dissertation\\code\\structures\\1SO0.pdb',
 'cleaned_pdb': 'C:\\Dissertation\\code\\structures\\1SO0_clean.pdb',
 'pockets': [{'name': 'pocket1',
   'center_x': 43.2839,
   'center_y': 10.3931,
   'center_z': 45.6117,
   'score': 13.68,
   'probability': 0.695},
  {'name': 'pocket2',
   'center_x': 10.5252,
   'center_y': 17.1927,
   'center_z': 3.1027,
   'score': 13.59,
   'probability': 0.692},
  {'name': 'pocket3',
   'center_x': 25.2976,
   'center_y': 44.557,
   'center_z': 2.1502,
   'score': 12.84,
   'probability': 0.664},
  {'name': 'pocket4',
   'center_x': -5.2618,
   'center_y': -7.9336,
   'center_z': -0.7031,
   'score': 12.75,
   'probability': 0.661},
  {'name': 'pocket5',
   'center_x': -2.6092,
   'center_y': 25.1808,
   'center_z': 48.5626,
   'score': 11.7,
   'probability': 0.622},
  {'name': 'pocket6',
   'center_x': 18.8673,
   'center_

In [ ]:
# ── Integrated: reads from Obj-1 output ──
# First run obj_1_kg_agent.ipynb:
#   run_kg_agent("Identify druggable targets in Alzheimer's disease")
# Then run this:
run_structure_agent_from_kg_output()

In [ ]:
# ── LLM explanation only (on existing docking results) ──
run_pocket_explanation_only("1V54", "P07471", protein_name="Alpha-1-antitrypsin")

In [16]:
run_structure_agent("P07471", protein_name="Alpha-1-antitrypsin")


════════════════════════════════════════════════════════════
  OBJ-2 STRUCTURE AGENT  |  Python 3.10.20 / meeko_env
  UniProt: P07471  |  Alpha-1-antitrypsin
════════════════════════════════════════════════════════════

📥 Downloading structure for P07471

🔍 Checking RCSB for P07471...
   Found 93 PDB entries — selecting best...
   1OCC: method=X-ray, res=2.8, ligands=['14', '15', '16', '17'], score=55
   1OCO: method=X-ray, res=2.8, ligands=['14', '15', '16', '17', '18', '19'], score=55
   1OCR: method=X-ray, res=2.35, ligands=['14', '15', '16', '17', '18'], score=60
   1OCZ: method=X-ray, res=2.9, ligands=['14', '15', '16', '17', '18', '19'], score=55
   1V54: method=X-ray, res=1.8, ligands=['14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27'], score=65
   1V55: method=X-ray, res=1.9, ligands=['14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27'], score=65
   2DYR: method=X-ray, res=1.8, ligands=['14', '15', '16', '17', '18

{'raw_query': 'P07471',
 'uniprot_id': 'P07471',
 'protein_name': 'Alpha-1-antitrypsin',
 'pdb_id': '1V54',
 'pdb_file': 'C:\\Dissertation\\code\\structures\\1V54.pdb',
 'cleaned_pdb': 'C:\\Dissertation\\code\\structures\\1V54_clean.pdb',
 'pockets': [{'name': 'pocket1',
   'center_x': 55.2788,
   'center_y': 302.5667,
   'center_z': 197.3137,
   'score': 280.64,
   'probability': 1.0},
  {'name': 'pocket2',
   'center_x': 132.5075,
   'center_y': 319.516,
   'center_z': 193.2643,
   'score': 270.93,
   'probability': 1.0},
  {'name': 'pocket3',
   'center_x': 113.3055,
   'center_y': 293.2965,
   'center_z': 201.5825,
   'score': 121.7,
   'probability': 0.998},
  {'name': 'pocket4',
   'center_x': 75.4648,
   'center_y': 327.2603,
   'center_z': 205.1647,
   'score': 119.61,
   'probability': 0.998},
  {'name': 'pocket5',
   'center_x': 50.5993,
   'center_y': 289.1098,
   'center_z': 186.5093,
   'score': 28.6,
   'probability': 0.926},
  {'name': 'pocket6',
   'center_x': 135.9198,